# cAPTure: XGB-P sanity controls

This CPU-only notebook audits the completed depth-5 XGB-P development run. It does not alter the primary models or OOF scores, access test scenarios, select thresholds, or choose new model features. The shuffled-label negative control uses a deterministic one-in-100 packet sample to keep the check practical on Colab CPU; it is not a replacement for full-data validation.

## 1. Mount Drive and load the repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/utils/capture_xgb_p_sanity.py", "code/python/tests/test_capture_xgb_p_sanity.py", "code/python/requirements-capture-xgb.txt"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("XGB-P sanity environment is ready.")
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

## 2. Run synthetic checks

In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for pattern in ("test_capture_xgb_p.py", "test_capture_xgb_p_sanity.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "code/python/tests"), "-p", pattern, "-v"], env=test_environment, cwd=PROJECT_ROOT, check=True)
print("Synthetic XGB-P checks passed.")

## 3. Bind the completed development artifacts

Set `RUN_ID` to an existing sanity run ID when resuming. The baseline run is read-only. Its archived manifest predates the newly frozen context-feature proposal, so the sanity runner checks the baseline configuration, preprocessing hash, scenario assignments, and prepared packet hashes rather than requiring identical manifest-file hashes.

In [ ]:
import pandas as pd
from IPython.display import display
from utils.capture_data import load_manifest, sha256_file
from utils.capture_xgb_p_sanity import (
    run_shuffled_label_control,
    run_single_feature_diagnostics,
    summarize_shuffled_label_controls,
    validate_shuffled_label_control,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
RUN_ID = None  # Replace with an earlier sanity run ID only when resuming.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p_sanity"
SANITY_RUN_DIR = DRIVE_ROOT / "xgb_p_sanity_runs" / RUN_ID
BATCH_SIZE = 100_000
CPU_THREADS = 2
for path in (PREPARED_RUN_DIR, PREPROCESSING_AUDIT_DIR, BASELINE_RUN_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required Drive run is missing: {path}")
manifest = load_manifest(MANIFEST_PATH)
print("Baseline run:", BASELINE_RUN_DIR)
print("Sanity output:", SANITY_RUN_DIR)
print("Negative-control sample modulus:", manifest["training"]["xgb_p_sanity_controls"]["shuffled_training_labels"]["sample_modulus"] )

## 4. Inspect fixed single-feature diagnostics

These are descriptive held-out ROC-AUC values for declared individual packet features. The best-direction value uses the validation labels to describe scalar ranking separation and must not be used to select features or tune the model. A nonlinear one-feature tree could behave differently.

In [ ]:
feature_report_path = SANITY_RUN_DIR / "single_feature_diagnostics.json"
if feature_report_path.exists():
    feature_report = json.loads(feature_report_path.read_text(encoding="utf-8"))
    if feature_report.get("status") != "single_feature_oof_diagnostics_complete" or feature_report.get("manifest_sha256") != sha256_file(MANIFEST_PATH) or feature_report.get("code_sha256") != sha256_file(PROJECT_ROOT / "code/python/utils/capture_xgb_p_sanity.py"):
        raise ValueError("Existing single-feature diagnostics do not match the current manifest.")
else:
    feature_report = run_single_feature_diagnostics(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, baseline_run_dir=BASELINE_RUN_DIR, output_path=feature_report_path, batch_size=BATCH_SIZE)
feature_table = pd.DataFrame(feature_report["rows"])
display(feature_table.sort_values(["scenario", "best_direction_roc_auc_diagnostic"], ascending=[True, False]))
print("Feature diagnostics saved:", feature_report_path)

## 5. Run the sampled shuffled-label negative control

Each fold trains a separate depth-5, 200-round XGBoost model on its sampled training scenarios after a within-scenario label permutation. Validation uses the original labels and the same deterministic row sample. A complete existing fold is verified; incomplete outputs are not overwritten.

In [ ]:
def run_or_verify_control(fold):
    output_dir = SANITY_RUN_DIR / "shuffled_labels" / f"fold_{fold}"
    if output_dir.exists():
        print(f"Verifying existing shuffled-label fold {fold}...")
        report = validate_shuffled_label_control(output_dir, fold)
        if report["manifest_sha256"] != sha256_file(MANIFEST_PATH) or report["code_sha256"] != sha256_file(PROJECT_ROOT / "code/python/utils/capture_xgb_p_sanity.py"):
            raise ValueError("Existing control was produced by a different manifest or trainer.")
        return report
    print(f"Training sampled shuffled-label fold {fold}...")
    return run_shuffled_label_control(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR, baseline_run_dir=BASELINE_RUN_DIR, output_dir=output_dir, fold=fold, batch_size=BATCH_SIZE, nthread=CPU_THREADS)

control_a = run_or_verify_control("A")
display(pd.DataFrame.from_dict(control_a["validation"], orient="index")[["sampled_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

In [ ]:
control_b = run_or_verify_control("B")
display(pd.DataFrame.from_dict(control_b["validation"], orient="index")[["sampled_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

## 6. Review the negative-control summary

A near-0.5 macro ROC-AUC is reassuring but does not prove the absence of leakage. A value above the predeclared 0.6 review line requires investigation before proceeding to XGB-P+T. The univariate diagnostics may reveal strong protocol fingerprints even if the label-shuffle control passes.

In [ ]:
control_summary = summarize_shuffled_label_controls(SANITY_RUN_DIR / "shuffled_labels")
display(pd.DataFrame.from_dict(control_summary["scenario_metrics"], orient="index")[["fold", "sampled_packets", "packet_roc_auc"]])
print("Shuffled-label hierarchical macro ROC-AUC:", control_summary["hierarchical_macro_oof_packet_roc_auc"])
print("Investigation required:", control_summary["review_required"])
print("This is a sampled negative control, not a full-data model estimate.")